In [ ]:
# Cell 1 — Load config
%run /home/jovyan/work/setup/config.py
import sys; sys.path.insert(0, "/home/jovyan/work")
from utils.delta_utils import save_layer

In [ ]:
# Cell 2 — Build dim_geography (region + derived country)
from pyspark.sql.functions import xxhash64, col, when

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/silver_beverage_sales_enriched")

dim_geography = (
    df_silver
    .select("region")
    .dropDuplicates(["region"])
    .withColumn("geography_sk", xxhash64(col("region")))
    .withColumn("country", when(col("region") == "CANADA", "Canada")
                           .otherwise("United States"))
    .select("geography_sk", "region", "country")
)

save_layer(dim_geography, "dim_geography", GOLD_PATH, PG_WRITE_PROPS)
dim_geography.show()